In [ ]:
import pandas as pd

In [ ]:
import os

os.listdir()

['.config', 'processed_travel_dataset.csv', 'sample_data']

In [ ]:
import pandas as pd

df = pd.read_csv("processed_travel_dataset.csv")

In [ ]:
import os
print(os.getcwd())

/content


In [ ]:
os.listdir()

['.config', 'processed_travel_dataset.csv', 'sample_data']

In [ ]:
df.head()

,Zone,State,City,Name,Type,Establishment Year,time needed to visit in hrs,Google review rating,Entrance Fee in INR,Airport with 50km Radius,...,Year Known,Age Category,Budget Category,Duration Category,rating_scaled,reviews_scaled,fee_scaled,Student Score,Popularity Score,Tags
0,Northern,Delhi,Delhi,India Gate,War Memorial,1921.0,0.5,4.6,0,Yes,...,Known,Historic,Very Low,Quick Visit,0.914286,0.350474,0.000000,0.762285,5.892296,War Memorial Historical Very Low Quick Visit E...
1,Northern,Delhi,Delhi,Humayun's Tomb,Tomb,1572.0,2.0,4.5,30,Yes,...,Known,Heritage,Very Low,Quick Visit,0.885714,0.052774,0.004000,0.657889,1.514125,Tomb Historical Very Low Quick Visit Afternoon
2,Northern,Delhi,Delhi,Akshardham Temple,Temple,2005.0,5.0,4.6,60,Yes,...,Known,Modern,Low,Full Day,0.914286,0.052774,0.008000,0.671375,1.547772,Temple Religious Low Full Day Afternoon
3,Northern,Delhi,Delhi,Waste to Wonder Park,Theme Park,2019.0,2.0,4.1,50,Yes,...,Known,Modern,Very Low,Quick Visit,0.771429,0.035183,0.006667,0.594936,0.979969,Theme Park Environmental Very Low Quick Visit ...
4,Northern,Delhi,Delhi,Jantar Mantar,Observatory,1724.0,2.0,4.2,15,Yes,...,Known,Heritage,Very Low,Quick Visit,0.800000,0.040595,0.002000,0.611779,1.134114,Observatory Scientific Very Low Quick Visit Mo...


In [ ]:
df.columns

Index(['Zone', 'State', 'City', 'Name', 'Type', 'Establishment Year',
       'time needed to visit in hrs', 'Google review rating',
       'Entrance Fee in INR', 'Airport with 50km Radius', 'Weekly Off',
       'Significance', 'DSLR Allowed', 'Number of google review in lakhs',
       'Best Time to visit', 'Attraction Age', 'Year Known', 'Age Category',
       'Budget Category', 'Duration Category', 'rating_scaled',
       'reviews_scaled', 'fee_scaled', 'Student Score', 'Popularity Score',
       'Tags'],
      dtype='object')

In [ ]:
"Tags" in df.columns

True

In [ ]:
df["Tags"].head()

,Tags
0,War Memorial Historical Very Low Quick Visit E...
1,Tomb Historical Very Low Quick Visit Afternoon
2,Temple Religious Low Full Day Afternoon
3,Theme Park Environmental Very Low Quick Visit ...
4,Observatory Scientific Very Low Quick Visit Mo...


In [ ]:
df.columns.tolist()

NameError: name 'df' is not defined

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(df["Tags"])

In [ ]:
tfidf_matrix.shape

(325, 123)

In [ ]:
cosine_sim = cosine_similarity(tfidf_matrix)

In [ ]:
cosine_sim.shape

(325, 325)

In [ ]:
indices = pd.Series(
    df.index,
    index=df["Name"]
).drop_duplicates()

In [ ]:
def recommend_places(place_name, top_n=5):

    idx = indices[place_name]

    sim_scores = list(
        enumerate(cosine_sim[idx])
    )

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:top_n+1]

    place_indices = [
        i[0]
        for i in sim_scores
    ]

    recommendations = df.iloc[
        place_indices
    ][[
        "Name",
        "Type",
        "City",
        "Student Score"
    ]]

    return recommendations

    recommendations = df.iloc[place_indices]

In [ ]:
df["Name"].head(20)

,Name
0,India Gate
1,Humayun's Tomb
2,Akshardham Temple
3,Waste to Wonder Park
4,Jantar Mantar
5,Chandni Chowk
6,Lotus Temple
7,Red Fort
8,Agrasen ki Baoli
9,Sunder Nursery


In [ ]:
recommend_places("India Gate")

,Name,Type,City,Student Score
203,Kargil War Memorial,War Memorial,Kargil,0.685755
207,Dras War Memorial,War Memorial,Dras,0.685795
262,War Memorial,War Memorial,Visakhapatnam,0.659132
212,Kirti Mandir,Memorial,Porbandar,0.686526
239,Vivekananda Rock Memorial,Memorial,Kanyakumari,0.675283


In [ ]:
def recommend_by_preferences(
    attraction_type=None,
    budget=None,
    visit_time=None,
    top_n=5
):

    filtered = df.copy()

    if attraction_type:
        filtered = filtered[
            filtered["Type"].str.contains(
                attraction_type,
                case=False,
                na=False
            )
        ]

    if budget:
        filtered = filtered[
            filtered["Budget Category"] == budget
        ]

    if visit_time:
        filtered = filtered[
            filtered["Best Time to visit"] == visit_time
        ]

    return (
        filtered
        .sort_values(
            by="Student Score",
            ascending=False
        )
        .head(top_n)
    )

In [ ]:
recommend_by_preferences(
    attraction_type="Temple",
    budget="Low"
)

,Zone,State,City,Name,Type,Establishment Year,time needed to visit in hrs,Google review rating,Entrance Fee in INR,Airport with 50km Radius,...,Year Known,Age Category,Budget Category,Duration Category,rating_scaled,reviews_scaled,fee_scaled,Student Score,Popularity Score,Tags
2,Northern,Delhi,Delhi,Akshardham Temple,Temple,2005.0,5.0,4.6,60,Yes,...,Known,Modern,Low,Full Day,0.914286,0.052774,0.008,0.671375,1.547772,Temple Religious Low Full Day Afternoon


In [ ]:
df[
    (df["Type"].str.contains("Temple", case=False, na=False))
    &
    (df["Budget Category"] == "Low")
].shape

(1, 26)

In [ ]:
df[df["Type"].str.contains("Temple", case=False, na=False)][
    ["Name", "Budget Category"]
]

,Name,Budget Category
2,Akshardham Temple,Low
6,Lotus Temple,Very Low
20,Siddhivinayak Temple,Very Low
21,Mahalaxmi Temple,Very Low
31,ISKCON Temple Bangalore,Very Low
38,Birla Mandir,Very Low
45,Dakshineswar Kali Temple,Very Low
46,Kalighat Kali Temple,Very Low
67,Dwarkadhish Temple,Very Low
71,Somnath Temple,Very Low


In [ ]:
df["Tags"].head()

,Tags
0,War Memorial Historical Very Low Quick Visit E...
1,Tomb Historical Very Low Quick Visit Afternoon
2,Temple Religious Low Full Day Afternoon
3,Theme Park Environmental Very Low Quick Visit ...
4,Observatory Scientific Very Low Quick Visit Mo...


In [ ]:
def recommend_trip(
    interest,
    top_n=10
):

    results = df.copy()

    results["Match Score"] = 0

    # Interest Match
    results.loc[
        results["Tags"].str.contains(
            interest,
            case=False,
            na=False
        ),
        "Match Score"
    ] += 5

    # Add Student Score influence
    results["Final Score"] = (
        results["Match Score"]
        +
        results["Student Score"] * 5
    )

    recommendations = (
        results
        .sort_values(
            by="Final Score",
            ascending=False
        )
        .head(top_n)
    )

    return recommendations[
        [
            "Name",
            "Type",
            "City",
            "Student Score",
            "Final Score"
        ]
    ]

In [ ]:
recommend_trip("Historical")

,Name,Type,City,Student Score,Final Score
17,Gateway of India,Monument,Mumbai,0.802880,9.014402
0,India Gate,War Memorial,Delhi,0.762285,8.811425
111,Mysore Palace,Palace,Mysore,0.756892,8.784460
181,Taj Mahal,Mausoleum,Agra,0.746743,8.733716
86,Chittorgarh Fort,Fort,Chittorgarh,0.732801,8.664007
32,Charminar,Landmark,Hyderabad,0.727035,8.635174
89,Amber Fort,Fort,Jaipur,0.714963,8.574817
295,Diu Fort,Fort,Diu,0.705451,8.527257
226,Sun Temple,Temple,Konark,0.703650,8.518251
7,Red Fort,Fort,Delhi,0.702411,8.512055


In [ ]:
def recommend_trip_budget(
    interest,
    total_budget,
    top_n=10
):

    results = df.copy()

    results["Match Score"] = 0

    results.loc[
        results["Tags"].str.contains(
            interest,
            case=False,
            na=False
        ),
        "Match Score"
    ] += 5

    results["Final Score"] = (
        results["Match Score"]
        +
        results["Student Score"] * 5
    )

    recommendations = (
        results
        .sort_values(
            by="Final Score",
            ascending=False
        )
    )

    selected = []

    remaining_budget = total_budget

    for _, row in recommendations.iterrows():

        fee = row["Entrance Fee in INR"]

        if fee <= remaining_budget:

            selected.append(row)

            remaining_budget -= fee

        if len(selected) >= top_n:
            break

    return pd.DataFrame(selected)

In [ ]:
recommend_trip_budget(
    interest="Historical",
    total_budget=1000
)

,Zone,State,City,Name,Type,Establishment Year,time needed to visit in hrs,Google review rating,Entrance Fee in INR,Airport with 50km Radius,...,Budget Category,Duration Category,rating_scaled,reviews_scaled,fee_scaled,Student Score,Popularity Score,Tags,Match Score,Final Score
17,Western,Maharastra,Mumbai,Gateway of India,Monument,1924.0,1.0,4.6,0,Yes,...,Very Low,Quick Visit,0.914286,0.485792,0.000000,0.802880,7.019859,Monument Historical Very Low Quick Visit All,5,9.014402
0,Northern,Delhi,Delhi,India Gate,War Memorial,1921.0,0.5,4.6,0,Yes,...,Very Low,Quick Visit,0.914286,0.350474,0.000000,0.762285,5.892296,War Memorial Historical Very Low Quick Visit E...,5,8.811425
111,Southern,Karnataka,Mysore,Mysore Palace,Palace,1912.0,2.0,4.6,50,Yes,...,Very Low,Quick Visit,0.914286,0.336942,0.006667,0.756892,5.762710,Palace Historical Very Low Quick Visit All,5,8.784460
181,Central,Uttar Pradesh,Agra,Taj Mahal,Mausoleum,1632.0,2.0,4.6,50,Yes,...,Very Low,Quick Visit,0.914286,0.303112,0.006667,0.746743,5.421813,Mausoleum Historical Very Low Quick Visit Morning,5,8.733716
86,Northern,Rajasthan,Chittorgarh,Chittorgarh Fort,Fort,700.0,2.0,4.6,40,No,...,Very Low,Quick Visit,0.914286,0.255751,0.005333,0.732801,4.897669,Fort Historical Very Low Quick Visit All,5,8.664007
32,Southern,Telangana,Hyderabad,Charminar,Landmark,1591.0,1.0,4.5,25,Yes,...,Very Low,Quick Visit,0.885714,0.282815,0.003333,0.727035,5.091310,Landmark Historical Very Low Quick Visit Morning,5,8.635174
89,Northern,Rajasthan,Jaipur,Amber Fort,Fort,1592.0,2.0,4.6,100,Yes,...,Low,Quick Visit,0.914286,0.201624,0.013333,0.714963,4.214937,Fort Historical Low Quick Visit All,5,8.574817
295,Western,Daman and Diu,Diu,Diu Fort,Fort,1535.0,1.5,4.6,0,No,...,Very Low,Quick Visit,0.914286,0.161028,0.000000,0.705451,3.626904,Fort Historical Very Low Quick Visit Afternoon,5,8.527257
226,Eastern,Odisha,Konark,Sun Temple,Temple,1250.0,1.5,4.7,40,Yes,...,Very Low,Quick Visit,0.942857,0.110961,0.005333,0.703650,2.840285,Temple Historical Very Low Quick Visit All,5,8.518251
7,Northern,Delhi,Delhi,Red Fort,Fort,1648.0,2.0,4.5,35,Yes,...,Very Low,Quick Visit,0.885714,0.201624,0.004667,0.702411,4.123308,Fort Historical Very Low Quick Visit Afternoon,5,8.512055


In [ ]:
def attractions_needed(days):

    if days == 1:
        return 3

    elif days == 2:
        return 6

    elif days == 3:
        return 8

    else:
        return 10

In [ ]:
attractions_needed(2)

6

In [ ]:
!pip install google-generativeai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv()
genai.configure(
    api_key=os.getenv("GEMINI_API_KEY")
)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash")

response = model.generate_content(
    "Create a 1-day budget-friendly itinerary for Delhi."
)

print(response.text)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3417.41ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 11425.62ms


Here's a 1-day budget-friendly itinerary for Delhi, focusing on iconic sights, delicious local food, and efficient public transport (the Delhi Metro is your best friend!).

**Budget Goal:** Aim for **₹500 - ₹800** per person (excluding accommodation).

---

### **Delhi on a Dime: 1-Day Itinerary**

**Theme:** Old Delhi Charm, Spiritual Serenity & Iconic Landmarks

**Transportation:** Primarily Delhi Metro, short auto-rickshaw rides, and walking. Get a **Delhi Metro Tourist Card** or a regular smart card for convenience and slight discounts.

---

**Morning (7:00 AM - 12:00 PM): Old Delhi's Heartbeat**

*   **7:00 AM - 7:30 AM: Travel to Old Delhi**
    *   Take the **Delhi Metro (Yellow Line)** to **Lal Qila** or **Chandni Chowk** station.
    *   *Budget Tip:* Always use the Metro for long distances.

*   **7:30 AM - 8:30 AM: Jama Masjid (Free Entry)**
    *   Start your day at India's largest mosque. The sheer scale and historical significance are breathtaking.
    *   Climb one of t

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")